# Gaussowski Naiwny Klasyfikator Bayesa – praca własna

Zgodnie z **Naive Bayes praca.pdf** (wzorzec: **Naive Bayes Jup Not demo.pdf**, bez odtwarzania dema):

1. Import bibliotek Pythona  
2. Import zbioru `VLagun_Phys_Years3.csv` (Zalew Wiślany, zmienna docelowa `Years`)  
3. Założenie: rozkład cech w klasach jest **gaussowski** (normalny)  
4. **Test hipotezy** o rozkładzie normalnym dla **dwóch wybranych zmiennych**  
5. Model **GaussianNB**, klasyfikacja probabilistyczna (`predict_proba`)  
6. **Powtórzenie** procedury dla **innej pary** dwóch zmiennych  

Zmienne w tej pracy: para 1 → `SS`, `TPOC`; para 2 → `Depth`, `PSU`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats
from matplotlib.patches import Ellipse

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid')
np.random.seed(42)

base_dir = Path.cwd()
data_path = base_dir / 'data' / 'VLagun_Phys_Years3.csv'
target_col = 'Years'

print('Katalog roboczy:', base_dir)
print('Plik danych:', data_path)

## 1. Wczytanie danych

In [ ]:
df = pd.read_csv(data_path)

X_all = df.iloc[:, :-1]
y_all = df.iloc[:, -1]

display(df.head())
print('Ksztalt:', df.shape)
print('Rozklad klas Years:')
print(y_all.value_counts().sort_index())

In [ ]:
eda_cols = ['SS', 'TPOC', 'Depth', 'PSU', target_col]
sns.pairplot(df[eda_cols], hue=target_col, corner=True, diag_kind='hist')
plt.suptitle('Podglad rozkladow i separacji klas (Years)', y=1.02)
plt.show()

## 2. Funkcje: test normalnosci i analiza GaussianNB

In [ ]:
def test_gaussian_hypothesis(dataframe, feature_pair, target_col='Years', alpha=0.05):
    """Test Shapiro-Wilka: H0 – dane pochodza z rozkladu normalnego (w obrebie klasy)."""
    rows = []
    fig, axes = plt.subplots(2, len(feature_pair), figsize=(5 * len(feature_pair), 8))
    if len(feature_pair) == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for j, feat in enumerate(feature_pair):
        for cls in sorted(dataframe[target_col].unique()):
            values = dataframe.loc[dataframe[target_col] == cls, feat].dropna().values
            if len(values) < 3:
                continue
            stat, p_value = stats.shapiro(values)
            rows.append({
                'cecha': feat,
                'klasa_Years': cls,
                'n': len(values),
                'statystyka_W': stat,
                'p_value': p_value,
                'normalnosc_alfa_0.05': 'TAK' if p_value > alpha else 'NIE',
            })

        for i, cls in enumerate(sorted(dataframe[target_col].unique())):
            values = dataframe.loc[dataframe[target_col] == cls, feat].dropna()
            sns.histplot(values, kde=True, ax=axes[0, j], stat='density', label=f'Years={cls}', alpha=0.5)
            stats.probplot(values, dist='norm', plot=axes[1, j])
        axes[0, j].set_title(f'Histogram + KDE: {feat}')
        axes[1, j].set_title(f'Q-Q plot: {feat}')
        axes[0, j].legend()

    plt.suptitle('Test hipotezy o rozkladzie gaussowskim (normalnym)', y=1.01)
    plt.tight_layout()
    plt.show()

    result = pd.DataFrame(rows)
    display(result.round(4))
    return result


def plot_gaussian_ellipses(ax, model, feat_x_idx, feat_y_idx, n_std=2.0):
    """Elipsy modelu generatywnego (srednia + wariancja w kazdej klasie)."""
    colors = ['tab:blue', 'tab:orange', 'tab:green']
    for i, cls in enumerate(model.classes_):
        mean_x = model.theta_[i, feat_x_idx]
        mean_y = model.theta_[i, feat_y_idx]
        std_x = np.sqrt(model.var_[i, feat_x_idx])
        std_y = np.sqrt(model.var_[i, feat_y_idx])
        ell = Ellipse(
            (mean_x, mean_y),
            width=2 * n_std * std_x,
            height=2 * n_std * std_y,
            facecolor=colors[i % len(colors)],
            edgecolor=colors[i % len(colors)],
            alpha=0.25,
            label=f'Gauss Years={cls}',
        )
        ax.add_patch(ell)


def run_gnb_for_pair(dataframe, feature_pair, target_col='Years', test_size=0.3, random_state=42):
    feat_x, feat_y = feature_pair
    X = dataframe[[feat_x, feat_y]].copy()
    y = dataframe[target_col].copy()

    print(f'\n{"=" * 60}')
    print(f'Para cech: {feat_x} i {feat_y}')
    print(f'{"=" * 60}')

    normality = test_gaussian_hypothesis(dataframe, feature_pair, target_col=target_col)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    model = GaussianNB()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f'\nAccuracy (test): {acc:.4f}')
    print('\nMacierz pomylek:')
    print(confusion_matrix(y_test, y_pred))
    print('\nClassification report:')
    print(classification_report(y_test, y_pred, digits=4))

    proba_cols = [f'P(Years={c})' for c in model.classes_]
    proba_df = pd.DataFrame(y_proba, columns=proba_cols)
    proba_df['y_true'] = y_test.to_numpy()
    proba_df['y_pred'] = y_pred

    print('\nKlasyfikacja probabilistyczna – predict_proba (przyklady):')
    display(proba_df.head(10).round(4))

    feat_x_idx = list(X.columns).index(feat_x)
    feat_y_idx = list(X.columns).index(feat_y)

    x_min, x_max = X[feat_x].min() - 1, X[feat_x].max() + 1
    y_min, y_max = X[feat_y].min() - 1, X[feat_y].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )
    grid = pd.DataFrame({feat_x: xx.ravel(), feat_y: yy.ravel()})
    zz = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(9, 6))
    plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
    plot_gaussian_ellipses(plt.gca(), model, feat_x_idx, feat_y_idx)
    sns.scatterplot(data=X_train, x=feat_x, y=feat_y, hue=y_train, palette='coolwarm', edgecolor='k', s=50, alpha=0.9)
    sns.scatterplot(
        data=X_test, x=feat_x, y=feat_y, hue=y_test, palette='coolwarm',
        marker='^', edgecolor='k', s=70, alpha=0.9, legend=False,
    )
    plt.title(f'GaussianNB: granica decyzyjna i elipsy Gaussa ({feat_x} vs {feat_y})')
    plt.legend(loc='best')
    plt.tight_layout()
    plt.show()

    return {
        'features': feature_pair,
        'accuracy': acc,
        'model': model,
        'normality': normality,
        'proba_table': proba_df,
    }

## 3. Para 1 – `SS` i `TPOC` (test hipotezy + klasyfikacja)

In [ ]:
result_1 = run_gnb_for_pair(df, feature_pair=('SS', 'TPOC'))

## 4. Para 2 – `Depth` i `PSU` (powtorzenie procedury)

In [ ]:
result_2 = run_gnb_for_pair(df, feature_pair=('Depth', 'PSU'))

## 5. Porownanie par i interpretacja

In [ ]:
summary = pd.DataFrame([
    {
        'para_cech': ', '.join(result_1['features']),
        'accuracy_test': result_1['accuracy'],
    },
    {
        'para_cech': ', '.join(result_2['features']),
        'accuracy_test': result_2['accuracy'],
    },
]).sort_values('accuracy_test', ascending=False)

display(summary.round(4))

best = summary.iloc[0]
display(Markdown(f"""
### Wnioski

- **Zmienna docelowa:** `Years` (klasy 0 i 1) – rozdzielenie lat obserwacji w Zalewie Wiślanym.
- **Zalozenie modelu:** w kazdej klasie cechy maja rozklad gaussowski (sprawdzony testem Shapiro-Wilka i wizualnie: histogram + Q-Q).
- **Klasyfikacja probabilistyczna:** `predict_proba` zwraca prawdopodobienstwa a posteriori P(Years=k | cechy).
- **Najlepsza para w tym zbiorze:** `{best['para_cech']}` (accuracy test = **{best['accuracy_test']:.4f}**).
- Elipsy na wykresie pokazuja gaussowski model generatywny dla kazdej etykiety (srednia i wariancja z `GaussianNB`).
"""))

## 6. Eksport HTML (tabele wynikow)

In [ ]:
html_path = base_dir / 'task.html'

html_body = f"""
<html><head><meta charset='utf-8'><title>Gaussian Naive Bayes – wyniki</title>
<style>body{{font-family:Arial,sans-serif;margin:24px}} table{{border-collapse:collapse;margin:12px 0}}
th,td{{border:1px solid #ccc;padding:6px}}</style></head><body>
<h1>Gaussowski Naiwny Klasyfikator Bayesa</h1>
<p>Zbior: VLagun_Phys_Years3.csv | Cel: Years</p>
<h2>Porownanie par cech</h2>
{summary.round(4).to_html(index=False)}
<h2>Test normalnosci – para SS, TPOC</h2>
{result_1['normality'].round(4).to_html(index=False)}
<h2>Test normalnosci – para Depth, PSU</h2>
{result_2['normality'].round(4).to_html(index=False)}
<h2>predict_proba (przyklad, SS/TPOC)</h2>
{result_1['proba_table'].head(10).round(4).to_html(index=False)}
</body></html>
"""

html_path.write_text(html_body, encoding='utf-8')
print('Zapisano:', html_path)
print('Notebook: task.ipynb')